# Working with SQL

The Arrow Flight Protocol was extended to be more SQL friendly with the  Arrow Flight SQL protocol.
However, Python never got a Arrow Flight SQL implementation, because that tends to be reserved for databases - don't build databases in Python!

Instead a lot of the effort went into building ADBC (Arrow Database Connectivity) to be a drop-in replacement between the client and the database,
translating the database wire protocol into Arrow instead of a format like ODBC

## The ODBC way

![ODBC replacement](images/db_odbc.png)

## The ADBC way
![ADBC](images/adbc_server.png)

Let's compare to regular SQLAlchemy

In [40]:
import sqlalchemy as sa
import polars as pl

# Predefine the SQL to run
SQL = "select * from messages limit 1000000"
engine = sa.create_engine("postgresql+psycopg://postgres:postgres@db:5432")

In [11]:
%%timeit -r 1
with engine.connect() as conn:
    records = conn.execute(sa.text(SQL))
    data = records.fetchall()
    pl.from_records(data, infer_schema_length=None)

4.62 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## ADBC

Now let's try with ADBC - The Postgres ADBC driver speaks the Postgres wire protocol and converts it directly to Arrow, before handing it to Polars.

This saves us a full conversion roundtrip!

In [18]:
from adbc_driver_postgresql.dbapi import connect

conn = connect("postgresql://postgres:postgres@db:5432")

In [15]:
%%timeit -r 1
with conn.cursor() as c:
    c.execute(SQL)
    pl.from_arrow(c.fetch_arrow_table())

2.64 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Connectorx
ADBC is not the only game in town for Arrow-based SQL. ConnectorX is a Rust-based database connector built for speed, which leverages Arrow for interoperability and compute.

In [19]:
import connectorx as cx

In [21]:
%%timeit -r 1
cx.read_sql("postgresql://postgres:postgres@db:5432", SQL, return_type="polars")

2.61 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


ConnectorX is interesting, because it lets us run multiple connections in parallel based on a partition column. This gives us the fastest speed so far in the test

In [45]:
%%timeit -r 1
cx.read_sql(
    "postgresql://postgres:postgres@db:5432",
    SQL,
    return_type="polars",
    partition_on="id",
    partition_num=10,
)

226 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Polars
Polars, being Arrow-native, supports both of these libraries as a DB backend.

In [31]:
import polars as pl

In [32]:
%%timeit -r 1
pl.read_database_uri(SQL, "postgresql://postgres:postgres@db:5432", engine="connectorx")

2.52 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [33]:
%%timeit -r 1
pl.read_database_uri(SQL, "postgresql://postgres:postgres@db:5432", engine="adbc")

2.71 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Polars also has the ability to work through "regular" SQLAlchemy, but there is no optimization involved

In [47]:
%%timeit -r 1
pl.read_database(SQL, engine, infer_schema_length=None)

4.85 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## DuckDB

Finally, what is a presentation about data without DuckDB being involved. DuckDB can natively attach to Postgres and federate queries.
As a demonstration, we load the full 10,000,000 row dataset in 12 seconds - 50% faster than "just" SQLAlchemy.

In [35]:
import duckdb

conn = duckdb.connect()

conn.install_extension("postgres")
conn.load_extension("postgres")
conn.execute("ATTACH 'postgresql://postgres:postgres@db:5432' as pg (TYPE POSTGRES)")

In [39]:
%%timeit -r 1
pl.from_arrow(conn.execute("SELECT * FROM pg.messages").to_arrow_table())

12.6 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Conclusion

As ADBC support grows, and it is growing, there is not much reason to stick with older, row-oriented technologies like ODBC. With a simple drop-in replacement, we can get a 50% speed improvement on our data,
and with a bit of work, we saw in our demo 